In [8]:
# Qiskit: Create and manipulate quantum circuits.
# Reference: https://quantum.cloud.ibm.com/docs/guides/circuit-construction
from qiskit import QuantumCircuit

# Qiskit Aer: Simulate quantum circuits locally without real hardware.
# Reference: https://qiskit.github.io/qiskit-aer/
from qiskit_aer import AerSimulator

# IBM Quantum Runtime: Submit quantum jobs and access IBM hardware.
# Reference: https://quantum.cloud.ibm.com/docs/guides/run-jobs
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService

# Transpiler: Convert circuits into instructions compatible with the backend.
# Reference: https://quantum.cloud.ibm.com/docs/guides/transpile
from qiskit.transpiler import generate_preset_pass_manager

# JSON: Read credentials from a local JSON file.
# Reference: https://docs.python.org/3/library/json.html
import json

print("Imports loaded successfully — you're ready to go.")

Imports loaded successfully — you're ready to go.


In [9]:
# Load API credentials from a local JSON file.
# The file should contain two fields: "apikey" and "crn".
# Never share this file or upload it to GitHub.
with open("api.json", "r") as f:
    credentials = json.load(f)

api_key = credentials["apikey"]  # IBM Cloud API key
crn = credentials["crn"]         # Cloud Resource Name (instance identifier)

# Save credentials locally for future IBM Quantum sessions.
# channel: IBM Quantum Platform authentication.
# instance: Specifies the IBM Quantum service instance.
# token: API key used for authentication.
# set_as_default: Use these credentials by default.
# overwrite: Replace previously saved default credentials.
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    instance=crn,
    token=api_key,
    set_as_default=True,
    overwrite=True
)

# You only need to do this once per environment.
# Afterwards, simply use:
# service = QiskitRuntimeService()

# Reference: https://quantum.cloud.ibm.com/docs/guides/save-credentials

In [10]:
# Initialize the service using your configured credentials.
# Reference: https://quantum.cloud.ibm.com/docs/guides/initialize-account
service = QiskitRuntimeService()

# Select an available real quantum processor.
real_backend = service.least_busy(operational=True, simulator=False)
print("Selected backend:", real_backend.name)

Selected backend: ibm_quebec


In [11]:
# Create a Bell-state circuit:
# |00> -> (|00> + |11>) / sqrt(2)
qc = QuantumCircuit(2)
qc.h(0)          # Apply Hadamard gate
qc.cx(0, 1)      # Apply CNOT gate
qc.measure_all() # Measure both qubits

# Transpile the circuit into instructions supported by the hardware.
# Reference: https://quantum.cloud.ibm.com/docs/guides/transpile
pm = generate_preset_pass_manager(
    backend=real_backend,
    optimization_level=1
)
isa_qc = pm.run(qc)

# Submit the circuit using SamplerV2.
# shots=100 means the circuit is executed 100 times.
# Reference: https://quantum.cloud.ibm.com/docs/guides/run-jobs
sampler = SamplerV2(real_backend)
job = sampler.run([isa_qc], shots=100)

# Save this Job ID to check its status and retrieve results later.
print("Job ID:", job.job_id())

Job ID: dbnk7klt7fqc73ajpdcg


In [12]:
# Initialize service again if using a new Python session.
# service = QiskitRuntimeService()

# Replace with the Job ID printed in Step 1.
# job_id = "YOUR_JOB_ID"
job_id = "dbnk7klt7fqc73ajpdcg"

# Retrieve the submitted job without waiting for completion.
job = service.job(job_id)

# Check the current execution status.
# Common statuses include QUEUED, RUNNING, DONE, and ERROR.
print("Job status:", job.status())

# Reference: https://quantum.cloud.ibm.com/docs/guides/monitor-job

Job status: DONE


In [13]:
# Initialize service again if using a new Python session.
# service = QiskitRuntimeService()

# Use the same Job ID from Step 1.
# job_id = "YOUR_JOB_ID"
job_id = "dbnk7klt7fqc73ajpdcg"

# Retrieve the previously submitted job.
job = service.job(job_id)

# Fetch the results.
# Note: job.result() waits if execution is not yet complete.
result = job.result()

# Extract measurement counts from the first circuit.
counts = result[0].data.meas.get_counts()
print("Measurement counts:", counts)

# Expected ideal Bell-state outcomes:
# Approximately 50% "00" and 50% "11".
# Small numbers of "01" and "10" may occur due to hardware noise.

# Reference: https://quantum.cloud.ibm.com/docs/guides/run-jobs

Measurement counts: {'11': 46, '00': 54}
